# DSA 4060 – Practical Quiz: Personalized Movie Recommender

**Name:** Sean  
**Student ID:** 669648  
**Assigned User ID:** 9  

Working: the last two digits of my Student ID are 48, and 48 MOD 40 = 8, so the assigned User ID is 8 + 1 = **9**.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

STUDENT_ID = 669648
USER_ID = (int(str(STUDENT_ID)[-2:]) % 40) + 1
print("Assigned User ID:", USER_ID)

Assigned User ID: 9


## Task 1 – Load and Inspect the Data

In [2]:
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

print("movies.csv - first five rows")
display(movies.head())
print("ratings.csv - first five rows")
display(ratings.head())

movies.csv - first five rows


,movie_id,title,genres
0,1,Inception,Sci-Fi|Thriller
1,2,Interstellar,Sci-Fi|Drama
2,3,The Matrix,Sci-Fi|Action
3,4,Arrival,Sci-Fi|Drama
4,5,Edge of Tomorrow,Sci-Fi|Action


ratings.csv - first five rows


,user_id,movie_id,rating
0,1,2,3.5
1,1,3,4.5
2,1,5,3.5
3,1,6,3.5
4,1,11,3.0


In [3]:
print("movies.csv  :", movies.shape[0], "rows,", movies.shape[1], "columns")
print("ratings.csv :", ratings.shape[0], "rows,", ratings.shape[1], "columns")

movies.csv  : 36 rows, 3 columns
ratings.csv : 440 rows, 3 columns


In [4]:
print("Missing values in movies.csv:")
print(movies.isnull().sum())
print("\nMissing values in ratings.csv:")
print(ratings.isnull().sum())

Missing values in movies.csv:
movie_id    0
title       0
genres      0
dtype: int64

Missing values in ratings.csv:
user_id     0
movie_id    0
rating      0
dtype: int64


**Finding:** Neither dataset has any missing values, every column has a complete entry in all rows (36 movies and 440 ratings), so no cleaning was needed before moving on.

## Task 2 – Profile the Assigned User

In [5]:
user_ratings = (ratings[ratings['user_id'] == USER_ID]
               .merge(movies, on='movie_id', how='left')
               .sort_values('rating', ascending=False)
               .reset_index(drop=True))
print("Movies rated by User", USER_ID, ":", len(user_ratings))
display(user_ratings[['movie_id', 'title', 'genres', 'rating']])

Movies rated by User 9 : 13


,movie_id,title,genres,rating
0,1,Inception,Sci-Fi|Thriller,4.5
1,8,John Wick,Action|Thriller,4.5
2,2,Interstellar,Sci-Fi|Drama,4.0
3,5,Edge of Tomorrow,Sci-Fi|Action,3.5
4,3,The Matrix,Sci-Fi|Action,3.5
5,26,A Quiet Place,Horror|Thriller,3.5
6,7,The Dark Knight,Action|Crime,3.5
7,29,Sherlock Holmes,Mystery|Action,3.5
8,4,Arrival,Sci-Fi|Drama,3.0
9,21,Toy Story,Animation|Comedy,3.0


In [6]:
top3 = user_ratings.head(3)[['title', 'genres', 'rating']]
print("Top three highest-rated movies:")
display(top3)

Top three highest-rated movies:


,title,genres,rating
0,Inception,Sci-Fi|Thriller,4.5
1,John Wick,Action|Thriller,4.5
2,Interstellar,Sci-Fi|Drama,4.0


In [7]:
# Evidence for preferred genres: count and average rating per genre
g = user_ratings.assign(genre=user_ratings['genres'].str.split('|')).explode('genre')
genre_summary = (g.groupby('genre')['rating']
                 .agg(movies_rated='count', avg_rating='mean')
                 .sort_values(['avg_rating', 'movies_rated'], ascending=False))
display(genre_summary.round(2))

,movies_rated,avg_rating
genre,,
Thriller,3,4.17
Sci-Fi,5,3.70
Action,6,3.50
Crime,1,3.50
Horror,1,3.50
Mystery,1,3.50
Animation,1,3.00
Comedy,1,3.00
Drama,4,2.88


**Preferred genres:** User 9 leans towards **Sci-Fi, Thriller and Action**. Their two highest ratings (4.5) are Inception (Sci-Fi|Thriller) and John Wick (Action|Thriller), and Interstellar (Sci-Fi) comes next at 4.0. Thriller has the best average rating in the table above (about 4.17 across three movies) and Sci-Fi has the most movies rated with a solid average. In contrast, Romance (The Notebook, 2.5), Biography/Drama (Hidden Figures, 2.0) and Superhero (Black Panther, 2.5) all got low scores, so the user seems to prefer tense, fast-paced films over romantic or character-driven ones.

## Task 3 – Build the Recommendation Logic

**Step 1 – Liked movies (rating ≥ 4.0)**

In [8]:
liked = user_ratings[user_ratings['rating'] >= 4.0]
print("Movies liked by User", USER_ID, "(rating >= 4.0):")
display(liked[['movie_id', 'title', 'genres', 'rating']])

Movies liked by User 9 (rating >= 4.0):


,movie_id,title,genres,rating
0,1,Inception,Sci-Fi|Thriller,4.5
1,8,John Wick,Action|Thriller,4.5
2,2,Interstellar,Sci-Fi|Drama,4.0


**Step 2 – Genres to numbers.** I used `CountVectorizer` with a custom tokenizer that splits on `|`, so a genre like "Sci-Fi" stays as one feature instead of being broken up.

In [9]:
vectorizer = CountVectorizer(tokenizer=lambda s: s.split('|'), lowercase=False, token_pattern=None)
genre_matrix = vectorizer.fit_transform(movies['genres'])

print("Genre features:", list(vectorizer.get_feature_names_out()))
print("Matrix shape:", genre_matrix.shape)
display(pd.DataFrame(genre_matrix.toarray(),
                     columns=vectorizer.get_feature_names_out(),
                     index=movies['title']).head())

Genre features: ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Drama', 'Family', 'Horror', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Sports', 'Superhero', 'Thriller']
Matrix shape: (36, 16)


,Action,Adventure,Animation,Biography,Comedy,Crime,Drama,Family,Horror,Musical,Mystery,Romance,Sci-Fi,Sports,Superhero,Thriller
title,,,,,,,,,,,,,,,,
Inception,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1
Interstellar,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
The Matrix,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
Arrival,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0
Edge of Tomorrow,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0


**Step 3 – Cosine similarity.** I built a user taste profile by averaging the genre vectors of the liked movies, then compared that profile with every movie in the catalogue.

In [10]:
liked_idx = movies.index[movies['movie_id'].isin(liked['movie_id'])]
user_profile = np.asarray(genre_matrix[liked_idx].mean(axis=0))   # shape (1, n_genres)

movies['similarity'] = cosine_similarity(user_profile, genre_matrix).ravel()

# also keep similarity to each liked movie, used later to explain the results
liked_sim = cosine_similarity(genre_matrix, genre_matrix[liked_idx])
print("User profile (average liked-genre weights):")
display(pd.Series(user_profile.ravel(), index=vectorizer.get_feature_names_out())
        .sort_values(ascending=False).round(3).to_frame('weight').T)

User profile (average liked-genre weights):


,Sci-Fi,Thriller,Action,Drama,Biography,Comedy,Animation,Adventure,Family,Crime,Horror,Musical,Romance,Mystery,Sports,Superhero
weight,0.667,0.667,0.333,0.333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


**Step 4 – Exclude already-rated movies and return exactly five.**

In [11]:
rated_ids = set(user_ratings['movie_id'])
candidates = movies[~movies['movie_id'].isin(rated_ids)].copy()

# tie-break on movie_id so the result is reproducible
top5 = (candidates.sort_values(['similarity', 'movie_id'], ascending=[False, True])
        .head(5).reset_index(drop=True))

print("Recommended titles:")
for i, t in enumerate(top5['title'], 1):
    print(i, t)

# safety checks
assert len(top5) == 5
assert not set(top5['movie_id']) & rated_ids
print("\nCheck passed: 5 movies, none already rated by User", USER_ID)

Recommended titles:
1 Blade Runner 2049
2 Mission: Impossible - Fallout
3 Get Out
4 The Shawshank Redemption
5 Mad Max: Fury Road

Check passed: 5 movies, none already rated by User 9


## Task 4 – Explain the Recommendations

In [12]:
liked_titles = liked['title'].tolist()
liked_genres = set(g for gs in liked['genres'] for g in gs.split('|'))

rows = []
for rank, row in top5.iterrows():
    midx = movies.index[movies['movie_id'] == row['movie_id']][0]
    best = int(np.argmax(liked_sim[midx]))
    best_title = movies.loc[liked_idx[best], 'title']
    shared_with_best = set(row['genres'].split('|')) & set(movies.loc[liked_idx[best], 'genres'].split('|'))
    shared_taste = set(row['genres'].split('|')) & liked_genres
    reason = (f"Shares {', '.join(sorted(shared_with_best)) or 'no genre'} with liked movie "
              f"'{best_title}'; matches the user's liked genres ({', '.join(sorted(shared_taste))}). "
              f"Profile similarity = {row['similarity']:.2f}.")
    rows.append([rank + 1, row['title'], row['genres'], reason])

results = pd.DataFrame(rows, columns=['Rank', 'Recommended Movie', 'Genres', 'Reason'])
pd.set_option('display.max_colwidth', None)
display(results)

,Rank,Recommended Movie,Genres,Reason
0,1,Blade Runner 2049,Sci-Fi|Drama,"Shares Drama, Sci-Fi with liked movie 'Interstellar'; matches the user's liked genres (Drama, Sci-Fi). Profile similarity = 0.67."
1,2,Mission: Impossible - Fallout,Action|Thriller,"Shares Action, Thriller with liked movie 'John Wick'; matches the user's liked genres (Action, Thriller). Profile similarity = 0.67."
2,3,Get Out,Horror|Thriller,Shares Thriller with liked movie 'Inception'; matches the user's liked genres (Thriller). Profile similarity = 0.45.
3,4,The Shawshank Redemption,Drama,Shares Drama with liked movie 'Interstellar'; matches the user's liked genres (Drama). Profile similarity = 0.32.
4,5,Mad Max: Fury Road,Action|Adventure,Shares Action with liked movie 'John Wick'; matches the user's liked genres (Action). Profile similarity = 0.22.


## Task 5 – Critical Thinking

**Limitation:** My recommender only looks at genre labels, so it can't tell the difference between two movies that share the same genres. For example, a great Sci-Fi thriller and a weak one look identical to the model. It also ignores how the other 39 users rated things, and with only 3 liked movies for this user the taste profile is built from very little evidence.

**Improvement:** I would combine the genre features with collaborative filtering, for example finding users whose ratings are similar to User 9 and using their high ratings for movies User 9 hasn't seen. Adding extra content features such as director, cast or plot keywords would also help, and weighting the profile by the actual rating (so a 4.5 counts more than a 4.0) would make it more personalized.